# Módulo 1: Clasificación de Severidad en Siniestros Viales

## 1. Contexto y Objetivo
El objetivo de este notebook es construir y comparar modelos de Machine Learning supervisados para predecir si un siniestro vial resultará en **Gravedad** (lesionados o fallecidos) o **Solo Daños**. 

### 2. Control de Fuga de Información (Data Leakage)
Para garantizar que el modelo sea evaluable en tiempo real (al momento del reporte de la emergencia), se excluyen estrictamente:
- Variables posteriores al evento (`fallecido_foraneo`, número de heridos/fallecidos).
- Identificadores y datos geográficos brutos (`codigo`, texto libre de dirección).
- La variable target original (`severidad`).

### 3. Estrategia de Evaluación
Se implementa un **Split Temporal Out-of-Time**:
- **Entrenamiento:** Siniestros ocurridos entre 2016 y 2021.
- **Evaluación (Test):** Siniestros ocurridos en el año 2022 completo.

In [22]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

In [23]:
df = pd.read_csv('C:\\Users\\santa\\Desktop\\Cositas de Python\\UNI\\IA\\proyectoIA\\data\\clean\\siniestralidad_clean.csv')

C:\Users\santa\AppData\Local\Temp\ipykernel_14916\2175947294.py:1: DtypeWarning: Columns (0: codigo, 1: medio_reporte, 2: direccion_original, 3: direccion_norm) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:\\Users\\santa\\Desktop\\Cositas de Python\\UNI\\IA\\proyectoIA\\data\\clean\\siniestralidad_clean.csv')


1. Carga, Limpieza y Feature Engineering

In [24]:
# 1. Creación de Target Binario (1 = Grave, 0 = Solo daños)
df['grave'] = (~df['severidad'].isin(['Solo daños'])).astype(int)

# 2. Tratamiento de variables de Vehículos y manejo consciente de NaN
veh_cols = [c for c in df.columns if c.startswith('veh_')]
df['veh_sin_dato'] = df[veh_cols].isna().any(axis=1).astype(int)
for col in veh_cols:
  df[col] = df[col].fillna(False).astype(int)

# 3. Formato de booleanos
df['es_fin_semana'] = df['es_fin_semana'].astype(int)
df['es_festivo'] = df['es_festivo'].astype(int)

# 4. Transformaciones Temporales (Cíclica para Mes + One-Hot para Día)
df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
df = pd.get_dummies(
    df, columns=['dia_semana'], prefix='dia', drop_first=True, dtype=int
)

# 5. Selección de Matriz de Características (X) y Target (y)
feature_cols = (
    ['anio', 'mes_sin', 'mes_cos', 'es_fin_semana', 'es_festivo', 'veh_sin_dato']
    + veh_cols
    + [c for c in df.columns if c.startswith('dia_')]
)

X = df[feature_cols]
y = df['grave']

print(f'Total registros: {X.shape[0]} | Número de predictores: {X.shape[1]}')

Total registros: 124889 | Número de predictores: 20


2. Split Temporal y Escalado

In [25]:
mask_train = df['anio'] < 2022
mask_test = df['anio'] == 2022

X_tr_raw, y_tr = X[mask_train], y[mask_train]
X_te_raw, y_te = X[mask_test], y[mask_test]

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)

print(
    f'Entrenamiento (2016-2021): {X_tr.shape[0]} registros | % Graves:'
    f' {y_tr.mean():.1%}'
)
print(
    f'Evaluación Test (2022): {X_te.shape[0]} registros | % Graves:'
    f' {y_te.mean():.1%}'
)

Entrenamiento (2016-2021): 106037 registros | % Graves: 36.2%
Evaluación Test (2022): 9979 registros | % Graves: 45.2%


3. Entrenamiento y Evaluación Comparativa

In [26]:
modelos = {
    'Perceptrón': Perceptron(max_iter=1000, random_state=42),
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'k-NN (k=15)': KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    'SVM Lineal': LinearSVC(dual=False, random_state=42),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
    ),
    'MLP (Red Neuronal)': MLPClassifier(
        hidden_layer_sizes=(32, 16), max_iter=200, random_state=42
    ),
}

resultados = []

In [27]:
for nombre, model in modelos.items():
  model.fit(X_tr, y_tr)
  y_pred = model.predict(X_te)
  report = classification_report(y_te, y_pred, output_dict=True)

  try:
    if hasattr(model, 'predict_proba'):
      auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    elif hasattr(model, 'decision_function'):
      auc = roc_auc_score(y_te, model.decision_function(X_te))
    else:
      auc = np.nan
  except Exception:
    auc = np.nan

  resultados.append({
      'Modelo': nombre,
      'Accuracy': report['accuracy'],
      'Precision (Grave)': report['1']['precision'],
      'Recall (Grave)': report['1']['recall'],
      'F1-Score (Grave)': report['1']['f1-score'],
      'ROC-AUC': auc,
  })

df_res = pd.DataFrame(resultados).sort_values(
    by='F1-Score (Grave)', ascending=False
)
display(df_res)

,Modelo,Accuracy,Precision (Grave),Recall (Grave),F1-Score (Grave),ROC-AUC
6,MLP (Red Neuronal),0.754685,0.701974,0.795570,0.745847,0.789068
4,Árbol de Decisión,0.734543,0.704605,0.711628,0.708099,0.790894
2,k-NN (k=15),0.728029,0.713845,0.665781,0.688975,0.778838
5,Random Forest,0.710091,0.736857,0.558804,0.635596,0.793627
3,SVM Lineal,0.667702,0.788916,0.362569,0.496813,0.791127
0,Perceptrón,0.645155,0.697646,0.380731,0.492621,0.668491
1,Regresión Logística,0.661990,0.806989,0.332447,0.470902,0.792833


4. ENTRENAMIENTO Y EVALUACIÓN DE MODELOS

In [17]:
modelos = {
    'Perceptrón': Perceptron(max_iter=1000, random_state=42),
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'k-NN (k=15)': KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    'SVM Lineal': LinearSVC(dual=False, random_state=42),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
    ),
    'MLP (Red Neuronal)': MLPClassifier(
        hidden_layer_sizes=(32, 16), max_iter=200, random_state=42
    ),
}

resultados = []

In [18]:
for nombre, model in modelos.items():
  print(f'Entrenando {nombre}...')
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

Entrenando Perceptrón...
Entrenando Regresión Logística...
Entrenando k-NN (k=15)...
Entrenando SVM Lineal...
Entrenando Árbol de Decisión...
Entrenando Random Forest...
Entrenando MLP (Red Neuronal)...


Métricas

In [19]:
report = classification_report(y_test, y_pred, output_dict=True)
try:
    if hasattr(model, 'predict_proba'):
      auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    elif hasattr(model, 'decision_function'):
      auc = roc_auc_score(y_test, model.decision_function(X_test))
    else:
      auc = np.nan
except Exception:
    auc = np.nan

resultados.append({
      'Modelo': nombre,
      'Accuracy': report['accuracy'],
      'Precision (Grave)': report['1']['precision'],
      'Recall (Grave)': report['1']['recall'],
      'F1-Score (Grave)': report['1']['f1-score'],
      'ROC-AUC': auc,
  })

In [20]:
df_resultados = pd.DataFrame(resultados).sort_values(
    by='F1-Score (Grave)', ascending=False
)
print('\n==========================================')
print('  COMPARATIVA DE RENDIMIENTO (TEST 2022)  ')
print('==========================================')
print(df_resultados.to_string(index=False))


  COMPARATIVA DE RENDIMIENTO (TEST 2022)  
            Modelo  Accuracy  Precision (Grave)  Recall (Grave)  F1-Score (Grave)  ROC-AUC
MLP (Red Neuronal)  0.754685           0.701974         0.79557          0.745847 0.789068
